# Hero Analysis Notebook

Verifying the flattened Hero tables using rich visualization.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe

# Bootstrap & Session
bootstrap_spark_env()
spark = SparkSession.builder.appName("HeroAnalyzer").getOrCreate()

# Config
conf = utils.get_app_conf("generate_powers")
warehouse_root = os.path.join(conf.get_string("stage_root"), "warehouse")
print(f"Reading Warehouse: {warehouse_root}")

## 1. Hero Profiles (Intent)

In [ ]:
hero_profiles = spark.read.parquet(os.path.join(warehouse_root, "hero_profiles"))
display_scrollable_dataframe(hero_profiles.toPandas())

## 2. Hero Genomes (Root Node)

In [ ]:
hero_genes = spark.read.parquet(os.path.join(warehouse_root, "hero_genes"))
display_scrollable_dataframe(hero_genes.limit(10).toPandas())

## 3. The Unified Hero View
A Hero is defined by their **Profile** (Intent) and their **Genome** (Mechanics). 
The Genome File contains **1 Master Regulator Gene** which controls a **Network** of downstream genes.

In [ ]:
# Load Regulation to count network size
reg = spark.read.parquet(os.path.join(warehouse_root, "hero_gene_regulation"))
network_size = reg.groupBy("hero_name").count().withColumnRenamed("count", "network_size")

# Join Profile + Genome + Network Size
# Use aliases to handle ambiguous columns like 'ontology' present in both tables
p = hero_profiles.alias("p")
g = hero_genes.alias("g")
n = network_size.alias("n")

full_view = p.join(g, "hero_name", "left") \
    .join(n, "hero_name", "left") \
    .select(
        F.col("hero_name"), 
        F.col("p.ontology"), # Explicitly select from profile
        F.col("p.primary_seed"), 
        F.col("g.gene_id"), 
        F.col("g.mutation_class"),
        F.col("n.network_size"),
        F.col("g.confidence")
    )

display_scrollable_dataframe(full_view.orderBy("hero_name").toPandas())

## 4. Specific Hero Check: Bugs Bunny

In [ ]:
bugs = full_view.filter(F.col("hero_name") == "Bugs Bunny")
display_scrollable_dataframe(bugs.toPandas())